In [1]:
import os
import pandas as pd
import time

# 라이브러리 설치: uv add pydrive2
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# 인증 과정 (만료될 때마다 브라우저 열림)
gauth = GoogleAuth()
gauth.LocalWebserverAuth() 
drive = GoogleDrive(gauth)

def get_subfolder_list(parent_id):
    # 구글 드라이브에게 날릴 질문(Query)을 작성
    # '부모가 parent_id이고, 타입이 폴더이며, 휴지통에 있지 않은 것'
    query = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = False"
    
    file_list = drive.ListFile({'q': query}).GetList()
    
    sub_folders = []
    for f in file_list:
        sub_folders.append({
            'name': f['title'],
            'id': f['id']
        })
    return sub_folders

# 각 폴더 ID를 넣으면 하위 폴더들이 list_of_folders 리스트로 들어감
list_of_folders_tongsin = get_subfolder_list('1LOTaqxasQWNucSTTPn_yGYU7A2D5cDsr')
list_of_folders_card = get_subfolder_list('14u8q6c3z2ndtWlVpI3pOYiIxRLGWHNkS')
list_of_folders_sinyong = get_subfolder_list('1rSiC1tq3dq5MGllTGlzCUhZYB_cnzVrD')
list_of_folders_company = get_subfolder_list('1FQ_6TZ2BxXoAx4Lpg9VNMzj_vlbJCHqV')

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=641874606667-v76q7nr0oa6dgm71k4q9vntrapbaeroq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.


## 기업, 신용
- utf-8-sig
- astype(str) 제거
- on_bad_lines='skip' 제거

In [2]:
def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False
        
        # 인코딩 utf-8-sig로 지정
        enc = 'utf-8-sig'
        try:
            # df = pd.read_csv(csv_path, encoding=enc, on_bad_lines='skip')
            df = pd.read_csv(csv_path, encoding=enc)
            # df.astype(str).to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except:
            success = False
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except:
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)


### 기업

In [3]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/기업"

for folder_info in list_of_folders_company:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 5. 기업 이전 통계(nps_move_cnt) 작업 시작...

🚀 4. 신규 기업(new) 작업 시작...

🚀 3. 법인 기업(cnt) 작업 시작...


### 신용

In [4]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/신용"

for folder_info in list_of_folders_sinyong:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 9. 이동통계(대민)(WORK_STAT) 작업 시작...

🚀 8. 신용정보(대민)(DM_STAT) 작업 시작...

🚀 4. 전출(대민)(OUT STAT) 작업 시작...

🚀 3. 전입(대민)(IN STAT) 작업 시작...

🚀 7. 전이(경계)(CHAGE_L) 작업 시작...

🚀 6. 전이(CHANGE) 작업 시작...


## 통신 : 
- utf-8-sig 
- astype(str) 제거
- on_bad_lines='skip' 제거
- dtype 지정 (dtype=columns_types)
- engine='pyarrow' 추가
- 에러유발 가능성 큰 컬럼들 사후처리 추가

In [5]:
columns_types = {
    # 식별자 및 코드 (문자열 또는 카테고리)
    'ETL_YM': 'str',  # 기준 년월 : 숫자로 읽으면 앞자리가 사라질 위험이 있으므로 문자열 타입이 가장 안전
    'ETL_YMD': 'str',
    'DOW': 'category', # 요일 구분 --> category
    'SEX_CD': 'category',
    'AGE_GRP': 'category',
    'PURPOSE': 'category',
    'TRANS_GB': 'category',
    
    # 명칭 (문자열)
    'O_MEGA_NM': 'str', 'D_MEGA_NM': 'str',
    'O_CTY_NM': 'str', 'D_CTY_NM': 'str',
    'O_ADMI_NM': 'str', 'D_ADMI_NM': 'str',
    
    # 코드성 숫자 (문자열 권장)
    'O_ADMI_CD': 'str', 'D_ADMI_CD': 'str', # 행정동 코드 : 숫자로 보이지만 실제로는 '코드'이므로 str 타입으로 읽는 것이 표준
    'O_CTY_CD': 'str', 'D_CTY_CD': 'str',
    
    # 시간대 (작은 정수)
    'O_TIME_CD': 'int8',
    'D_TIME_CD': 'int8',
    
    # 수치 데이터 (실수 - 결측치 대응)
    'CNT': 'float32',
    'O_CENTER_X': 'float32',
    'O_CENTER_Y': 'float32',
    'D_CENTER_X': 'float32',
    'D_CENTER_Y': 'float32',
    'DISTANCE': 'float32',
    'CARBON_EMISSIONS': 'float32'
}

def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False

        # 인코딩 utf-8-sig로 지정
        enc = 'utf-8-sig'
        try:
            # 파일 읽기전 0행의 컬럼명만 받아오기
            actual_cols = pd.read_csv(csv_path, nrows=0).columns
            # print(actual_cols) # 0번째 행 컬럼 헤더 깨지지않고 멀쩡한지 확인

            # 실제 파일에 존재하는 컬럼만 골라내어 dtype에 전달
            current_dtypes = {k: v for k, v in columns_types.items() if k in actual_cols}
            df = pd.read_csv(csv_path, encoding=enc, dtype=current_dtypes, engine='pyarrow') # pyarrow 사용하면 속도 빨라짐

            # 에러 유발 가능성이 높은 수치형 컬럼들 사후 처리
            cols_to_fix = ['D_CENTER_Y', 'O_CENTER_Y', 'CNT', 'DISTANCE', 'CARBON_EMISSIONS']
            for col in cols_to_fix:
                if col in df.columns:
                    # 글자가 섞여있어도 NaN으로 바꾸며 float32로 변환 (메모리 절약)
                    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except Exception as e:
            print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except Exception as e:
        print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)

In [ ]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/통신"

for folder_info in list_of_folders_tongsin:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 T27 작업 시작...

🚀 T26 작업 시작...
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202502_성남시.csv
✅ 완료: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202502_성남시.csv
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202501_성남시.csv
✅ 완료: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202501_성남시.csv
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202512_성남시.csv
✅ 완료: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202512_성남시.csv
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202511_성남시.csv
✅ 완료: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202511_성남시.csv
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202510_성남시.csv
✅ 완료: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202510_성남시.csv
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202508_성남시.csv
✅ 완료: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202508_성남시.csv
📥 다운로드 및 변환 중: T26_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_202312_성남시

## 카드
- cp949
- astype(str) 제거
- on_bad_lines='skip' 제거
- engine='pyarrow' 추가
- sep='|' 추가

In [ ]:
def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False

        # 인코딩 cp949로 지정
        enc = 'cp949'
        try:
            df = pd.read_csv(csv_path, sep='|', encoding=enc, engine='pyarrow') # pyarrow 사용하면 속도 빨라짐
            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except Exception as e:
            print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except Exception as e:
        print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)


🚀 10. 매출(대민)(day) 작업 시작...

🚀 3. 가맹점 정보(대민)(mer_s) 작업 시작...
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2512_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2512_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2511_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2511_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2510_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2510_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2509_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2509_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2508_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2508_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2507_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2507_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2506_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2506_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2505_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2505_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2504_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2504_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2503_성남시.csv
cp949
✅ 완료: tbsh_gyeonggi_mer_s_2503_성남시.csv
📥 다운로드 및 

In [ ]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/카드"

for folder_info in list_of_folders_card:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)

In [ ]:
mer_s_2301 = pd.read_parquet('seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2409_성남시.parquet', engine='fastparquet')
mer_s_2301

,ta_ym|cty_rgn_no|admi_cty_no|blk_cd|card_tpbuz_cd|card_tpbuz_nm_1|card_tpbuz_nm_2|sale|mm_cnt|mer_cnt|fran_cnt|open_cnt|stop_cnt|close_cnt|open_11_cnt|open_12_23_cnt|open_24_35_cnt|open_36_47_cnt|open_48_59_cnt|open_60_cnt
0,202409|41135|41135510|100470|Q13|음식|커피/음료|A|77...
1,202409|41135|41135510|100470|R07|학문/교육|입시학원|A|...
2,202409|41135|41135680|101678|D06|소매/유통|스포츠/레져용...
3,202409|41135|41135680|101678|D06|소매/유통|스포츠/레져용...
4,202409|41135|41135680|101678|O03|여가/오락|일반스포츠|A...
...,...
58271,202409|41135|41135670|99971|D19|소매/유통|제조/도매|A|...
58272,202409|41135|41135670|99971|F06|생활서비스|세탁/가사서비스...
58273,202409|41135|41135670|99971|F10|생활서비스|차량관리/서비스...
58274,202409|41135|41135670|99971|Q03|음식|닭/오리요리|A|15...


In [ ]:
mer_s_2301.info()

<class 'pandas.DataFrame'>
RangeIndex: 58276 entries, 0 to 58275
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                          Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                          --------------  ----- 
 0   ta_ym|cty_rgn_no|admi_cty_no|blk_cd|card_tpbuz_cd|card_tpbuz_nm_1|card_tpbuz_nm_2|sale|mm_cnt|mer_cnt|fran_cnt|open_cnt|stop_cnt|close_cnt|open_11_cnt|open_12_23_cnt|open_24_35_cnt|open_36_47_cnt|open_48_59_cnt|open_60_cnt  58276 non-null  object
dtypes: object(1)
memory usage: 455.4+ KB


In [ ]:
tbsh_2301 = pd.read_parquet('seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2301_성남시.parquet')
tbsh_2301.info()

<class 'pandas.DataFrame'>
RangeIndex: 2276923 entries, 0 to 2276922
Data columns (total 12 columns):
 #   Column           Dtype
---  ------           -----
 0   ta_ymd           str  
 1   cty_rgn_no       str  
 2   admi_cty_no      str  
 3   card_tpbuz_cd    str  
 4   card_tpbuz_nm_1  str  
 5   card_tpbuz_nm_2  str  
 6   hour             str  
 7   sex              str  
 8   age              str  
 9   day              str  
 10  amt              str  
 11  cnt              str  
dtypes: str(12)
memory usage: 367.2 MB


In [ ]:
tbsh_2301_csv = pd.read_csv('seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2301_성남시.csv', encoding='cp949')
tbsh_2301_csv

,ta_ymd,cty_rgn_no,admi_cty_no,card_tpbuz_cd,card_tpbuz_nm_1,card_tpbuz_nm_2,hour,sex,age,day,amt,cnt
0,20230101,41131,41131510,D01,소매/유통,가전제품,4,M,5,7,9251291,2
1,20230101,41131,41131510,D01,소매/유통,가전제품,5,F,5,7,62392,4
2,20230101,41131,41131510,D01,소매/유통,가전제품,5,F,6,7,153920,2
3,20230101,41131,41131510,D01,소매/유통,가전제품,5,F,7,7,21515,2
4,20230101,41131,41131510,D01,소매/유통,가전제품,5,M,6,7,544320,4
...,...,...,...,...,...,...,...,...,...,...,...,...
2276918,20230131,41135,41135680,Y04,공공/기업/단체,단체,7,M,3,2,22575,5
2276919,20230131,41135,41135680,Y04,공공/기업/단체,단체,7,M,4,2,22124,5
2276920,20230131,41135,41135680,Y04,공공/기업/단체,단체,7,M,5,2,40167,2
2276921,20230131,41135,41135680,Y04,공공/기업/단체,단체,7,M,6,2,253645,5


In [ ]:
nps_move_cnt_2405 = pd.read_parquet('seongnam_data/기업/5. 기업 이전 통계(nps_move_cnt)/gg_corp5_nps_move_cnt_202405_성남시.parquet')
nps_move_cnt_2405

,stdr_ym,out_sido_nm,out_sigun_nm,out_admi_nm,in_sido,in_sigun_nm,in_admi_nm,comp_cn
0,202405,경기도,성남시 분당구,구미동,경기도,평택시,이충동,1
1,202405,경기도,성남시 분당구,삼평동,경기도,성남시 분당구,백현동,1
2,202405,경기도,성남시 분당구,야탑동,경기도,성남시 중원구,여수동,1
3,202405,경기도,성남시 분당구,운중동,경기도,성남시 수정구,고등동,1
4,202405,경기도,성남시 분당구,정자동,경기도,안양시 동안구,호계동,1
5,202405,경기도,성남시 중원구,여수동,경기도,성남시 수정구,고등동,1
6,202405,경기도,성남시 중원구,상대원동,경기도,오산시,가장동,1
7,202405,경기도,성남시 중원구,상대원동,경기도,하남시,덕풍동,1
8,202405,경기도,성남시 중원구,상대원동,경기도,용인시 수지구,상현동,1
9,202405,경기도,용인시 처인구,포곡읍,경기도,성남시 분당구,수내동,1
